# ACE Page Context Agent — Production

Registers `PageContextAgentModel` to Unity Catalog and deploys `ace-lending-context-endpoint`.

The page context agent answers questions about the loan the officer is currently viewing, using only the live structured data sent by the Blazor app. No Vector Search dependency.
`LLM_ENDPOINT` is set to Sonnet.

| Cell | Purpose |
|------|---------|
| 1 | Install `mlflow` |
| 2 | Central config — `LLM_ENDPOINT` = Sonnet |
| 3 | Register `PageContextAgentModel` pyfunc to Unity Catalog |
| 4 | Deploy `ace-lending-context-endpoint` |

> **Prod note:** Dependency check, notebook test functions, unit tests, and model verification cells removed.
> Wait for Cell 4 to print **"ace-lending-context-endpoint is READY"** before running `supervisor_agent.ipynb`.


In [ ]:
# ================================================================
# CELL 1 -- Install dependencies
# PURPOSE: Install the only dependency this agent needs: mlflow.
#
#
# Only mlflow is installed here -- this agent does NOT use:
#   - databricks-vectorsearch  (no policy index lookups)
#   - presidio / spacy         (PII masking handled by Supervisor)
#
# mlflow is needed for:
#   - mlflow.set_experiment()        track test runs in Cell 5
#   - mlflow.pyfunc.PythonModel      base class for the registered model
#   - mlflow.pyfunc.log_model()      register to Unity Catalog in Cell 7
#   - mlflow.pyfunc.load_model()     load champion model in Cell 8
#
# After %pip install the kernel restarts automatically.
# Re-run from Cell 3 onwards -- do not re-run this cell again.
# ================================================================

%pip install mlflow
dbutils.library.restartPython()

In [ ]:
# ================================================================
# CELL 2 -- Config
# PURPOSE: Central config -- LLM endpoint, token budget, page context limit, system prompt, and model registry path.
#
#
# All tunable settings and the system prompt live here.
# Change values in this cell and re-run Cell 5 (interactive test)
# to evaluate quality without re-registering the model.
#
# Key settings:
#   LLM_ENDPOINT      : Claude Haiku 4.5 -- fast and cost-effective
#                       for structured data Q&A where sub-agents have
#                       already extracted all relevant facts.
#   MAX_TOKENS        : 768 -- enough for the mandated response
#                       format (Answer + Data Referenced +
#                       optional Calculation). Raise if answers
#                       are being truncated.
#   MAX_PAGE_CONTEXTS : 3 -- max pages per request. The Blazor
#                       package sends the current page + up to 2
#                       related tabs. Capped to control token
#                       budget.
#
# SYSTEM_PROMPT rules summary (11 rules):
#   Rule 1  -- never invent values  (hallucination prevention)
#   Rule 2  -- "As per [Tab] tab, ..." attribution
#   Rule 3  -- exact numbers/dates/names (no rounding or reformatting)
#   Rule 4  -- exact phrase for missing fields
#   Rule 5  -- exact phrase for conflicting values across tabs
#   Rule 6  -- calculations only when needed, show steps
#   Rule 7  -- report only, do not assess policy compliance
#   Rule 8  -- decline off-topic questions
#   Rule 9  -- prior conversation resolves references only
#   Rule 10 -- no unsupported statements
#   Rule 11 -- group tab citations for consecutive same-tab statements
#
# REGISTERED_MODEL is the Unity Catalog FQDN for the model:
#   ace_theorem.chat.page_context_agent
# ================================================================

CATALOG = "ace_theorem"
SCHEMA = "chat"
LLM_ENDPOINT = "databricks-claude-sonnet-4-5"
MAX_TOKENS = 768
MAX_PAGE_CONTEXTS = 3

EXPERIMENT_NAME  = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/ace-page-context-agent"
REGISTERED_MODEL = f"{CATALOG}.{SCHEMA}.page_context_agent"
MODEL_ALIAS      = "champion"

SYSTEM_PROMPT = """You are a lending data analyst for Vantage Bank. Your role is to answer questions about loan applications using only the structured page data the lending officer is currently viewing.

## Rules
1. Use only values explicitly present in the provided page data — never invent, estimate, or infer values not shown.
2. Attribute every factual statement to its source tab using: "As per [Tab] tab, ..."
3. Reproduce all numbers, amounts, percentages, dates, names, identifiers, and statuses exactly as provided — no rounding or reformatting.
4. If a value is not present, say exactly: "That field is not available on the current page(s). It may be on a different section of this application."
5. If the same field appears with different values across tabs, say exactly: "The current page data contains conflicting values for this field." Then list each value with its tab source.
6. Perform calculations only when asked or needed to answer directly, using only provided values — show steps clearly.
7. Do not assess policy compliance or underwriting risk — do not make recommendations. Report only what the data shows.
8. If the question is unrelated to the current loan page data, decline to answer.
9. Use prior conversation only to resolve references in the current question — never let prior conversation override the page data.
10. Do not include unsupported statements.
11. If consecutive statements reference the same tab, cite once at the end of the last statement — do not repeat "As per [Tab] tab" on every line.
12. For workflow navigation questions (available actions, routing, next owner, stage purpose, queue), use the Workflow_* fields from the Stage Gate Check tab as the authoritative source. Available fields: Workflow_Queue, Workflow_AssignedRole, Workflow_StagePurpose, Workflow_Activity, Workflow_AuthorityRequired, Workflow_Action_N_Name, Workflow_Action_N_Description, Workflow_Action_N_NextStage, Workflow_Action_N_NextOwner. Cite these as "As per Stage Gate Check tab, ..."
13. When answering "where does this go next?" or "what does [Action] do?", cite the specific Workflow_Action_N_* fields by action name. If Workflow_StagePurpose contains the phrase "has not been disclosed", say exactly: "Information for this workflow stage has not been disclosed. Please contact your workflow coordinator for details." If Workflow_* fields are absent from the page data entirely, say exactly: "Workflow routing details are not available on the current page. Please consult the workflow coordinator."
14. Workflow guidance is descriptive only. Never instruct the user to take a workflow action, trigger a workflow event, or click any button. Describe what an action would do; do not recommend whether the user should take it.

## Response format

**Answer**
- Direct answer in 1–2 sentences.

**Data Referenced**
- Field: exact value (as per [Tab] tab)

**Calculation** *(omit if none)*
- Step-by-step working using only provided values.
- Final result: exact result

**Limitations / Missing Data** *(omit if none)*
- [Field Name]: not available on current page(s)

**Conflicting Data** *(omit if none)*
- [Field Name]: [Value] (as per [Tab] tab)
- [Field Name]: [Value] (as per [Tab] tab)
"""
print(f"Config loaded. Model: {REGISTERED_MODEL}")

In [ ]:
# ================================================================
# CELL 3 -- Register PageContextAgentModel as an MLflow pyfunc
# PURPOSE: Register PageContextAgentModel to Unity Catalog as the production pyfunc.
#
#
# PageContextAgentModel is a self-contained MLflow PythonModel.
# All constants (LLM_ENDPOINT, MAX_TOKENS, SYSTEM_PROMPT) and all
# logic (_format_page, predict) live inside the class so the model
# works in a Databricks Model Serving container where no notebook
# globals are available.
#
# Class methods:
#   load_context(context)  -- called once when the container
#     starts. Initialises the mlflow deployments client.
#
#   _format_page(page)  -- converts one page context dict into
#     the structured text block sent to the LLM.
#
#   predict(context, model_input)  -- called on every request.
#     Accepts both pandas DataFrame (MLflow server) and plain
#     dict (notebook / unit test) for input flexibility.
#     Deserialises page_contexts JSON string -> list internally.
#     Returns answer + pages_used as JSON strings (MLflow schema
#     safe -- list fields cause infer_signature issues on some
#     MLflow versions).
#
# Why no SP credentials or Vector Search?
#   This agent only calls the LLM endpoint. The Model Serving
#   container automatically injects a DATABRICKS_TOKEN (M2M token)
#   that has access to the LLM endpoint via the resources= block.
#   No extra env vars or secret scopes needed.
#
# resources= block:
#   Declares DatabricksServingEndpoint("databricks-claude-haiku-4-5")
#   so the auto-injected M2M token has permission to call it.
#
# After this cell:
#   - Model is logged to MLflow under EXPERIMENT_NAME
#   - Model is registered as ace_theorem.chat.page_context_agent
#   - The latest version is aliased as "champion"
# ================================================================

import json as _json
import time
import mlflow
from mlflow.models import infer_signature
from mlflow.models.resources import DatabricksServingEndpoint

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)


class PageContextAgentModel(mlflow.pyfunc.PythonModel):

    # -- Constants ------------------------------------------------
    LLM_ENDPOINT      = "databricks-claude-sonnet-4-5"
    MAX_TOKENS        = 768
    MAX_PAGE_CONTEXTS = 3

    SYSTEM_PROMPT = """You are a lending data analyst for Vantage Bank. Your role is to answer questions about loan applications using only the structured page data the lending officer is currently viewing.

## Rules
1. Use only values explicitly present in the provided page data — never invent, estimate, or infer values not shown.
2. Attribute every factual statement to its source tab using: "As per [Tab] tab, ..."
3. Reproduce all numbers, amounts, percentages, dates, names, identifiers, and statuses exactly as provided — no rounding or reformatting.
4. If a value is not present, say exactly: "That field is not available on the current page(s). It may be on a different section of this application."
5. If the same field appears with different values across tabs, say exactly: "The current page data contains conflicting values for this field." Then list each value with its tab source.
6. Perform calculations only when asked or needed to answer directly, using only provided values — show steps clearly.
7. Do not make a compliance verdict or underwriting recommendation -- stating whether a requirement is met or not met is the synthesis layer's responsibility. However, when the question is a compliance or policy question, proactively surface ALL quantitative and categorical fields that policy criteria typically evaluate: loan amounts, collateral types and values, LTV ratios, DTI ratios, dates, environmental statuses, approval levels, and any other fields relevant to the question. Surface the data -- do not judge it.
8. If the question is unrelated to the current loan page data, decline to answer.
9. Use prior conversation only to resolve references in the current question — never let prior conversation override the page data.
10. Do not include unsupported statements.
11. If consecutive statements reference the same tab, cite once at the end of the last statement — do not repeat "As per [Tab] tab" on every line.
12. When the question involves a policy threshold or eligibility criterion, explicitly report every field the policy would evaluate -- even if the question does not name those fields directly. For example, if asked about environmental requirements, report loan amount, collateral type, and Phase I Assessment status from the page data even if not explicitly requested.
13. For workflow navigation questions (available actions, routing, next owner, stage purpose, queue), use the Workflow_* fields from the Stage Gate Check tab as the authoritative source. Available fields: Workflow_Queue, Workflow_AssignedRole, Workflow_StagePurpose, Workflow_Activity, Workflow_AuthorityRequired, Workflow_Action_N_Name, Workflow_Action_N_Description, Workflow_Action_N_NextStage, Workflow_Action_N_NextOwner. Cite these as "As per Stage Gate Check tab, ..."
14. When answering "where does this go next?" or "what does [Action] do?", cite the specific Workflow_Action_N_* fields by action name. If Workflow_StagePurpose contains the phrase "has not been disclosed", say exactly: "Information for this workflow stage has not been disclosed. Please contact your workflow coordinator for details." If Workflow_* fields are absent from the page data entirely, say exactly: "Workflow routing details are not available on the current page. Please consult the workflow coordinator."
15. Workflow guidance is descriptive only. Never instruct the user to take a workflow action, trigger a workflow event, or click any button. Describe what an action would do; do not recommend whether the user should take it.

## Response format

**Answer**
- Direct answer in 1–2 sentences.

**Data Referenced**
- Field: exact value (as per [Tab] tab)

**Calculation** *(omit if none)*
- Step-by-step working using only provided values.
- Final result: exact result

**Limitations / Missing Data** *(omit if none)*
- [Field Name]: not available on current page(s)

**Conflicting Data** *(omit if none)*
- [Field Name]: [Value] (as per [Tab] tab)
- [Field Name]: [Value] (as per [Tab] tab)
"""

    # -- load_context: runs once when the container starts --------
    def load_context(self, context):
        import os as _os, mlflow.deployments
        from msal import ConfidentialClientApplication
        self.client = mlflow.deployments.get_deploy_client("databricks")
        _tenant = _os.environ.get("SP_TENANT_ID", "")
        _cid    = _os.environ.get("SP_CLIENT_ID", "")
        _csec   = _os.environ.get("SP_CLIENT_SECRET", "")
        _has_sp = bool(_tenant and _cid and _csec)
        if _has_sp:
            self._msal_app = ConfidentialClientApplication(
                client_id=_cid,
                client_credential=_csec,
                authority=f"https://login.microsoftonline.com/{_tenant}"
            )
        else:
            self._msal_app = None
        print(f"[ACE] sp_vars={_has_sp}")

    # -- _get_token: scaffolded for future SP auth calls ----------
    def _get_token(self):
        """Return a valid SP OAuth token via MSAL (cached + auto-refreshed).
        Falls back to DATABRICKS_TOKEN for notebook/dev context.
        Not actively called by this agent today -- future-proofing.
        """
        import os as _os
        if self._msal_app is not None:
            result = self._msal_app.acquire_token_for_client(
                scopes=["2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default"]
            )
            if "access_token" in result:
                return result["access_token"]
        return (
            _os.environ.get("DATABRICKS_TOKEN", "") or
            _os.environ.get("DB_TOKEN", "")
        )

    # -- Helper: format a single page context block for the LLM --
    def _format_page(self, page):
        label = page.get("label", "Unknown Page")
        tabs  = page.get("tabs", {})
        lines = [
            "=" * 50,
            f"PAGE: {label}",
            "=" * 50,
        ]
        if tabs:
            for tab_name, tab_data in tabs.items():
                if not isinstance(tab_data, dict):
                    continue
                struct = tab_data.get("structured_bucket", {})
                chunks = tab_data.get("semantic_chunks", [])
                if not struct and not chunks:
                    continue
                lines.append(f"\n[{tab_name}]")
                if struct:
                    lines.append("  Structured Fields:")
                    for key, value in struct.items():
                        lines.append(f"    {key}: {value}")
                if chunks:
                    lines.append("  Additional Context:")
                    for chunk in chunks:
                        lines.append(f"    - {chunk}")
        else:
            struct = page.get("structured_bucket", {})
            chunks = page.get("semantic_chunks", [])
            if struct:
                lines.append("Structured Fields:")
                for key, value in struct.items():
                    lines.append(f"  {key}: {value}")
            if chunks:
                lines.append("\nAdditional Context:")
                for chunk in chunks:
                    lines.append(f"  - {chunk}")
        return "\n".join(lines)

    # -- predict: called on every request -------------------------
    def predict(self, context, model_input):
        import json as _json

        if isinstance(model_input, dict):
            question      = model_input["question"]
            page_contexts = model_input.get("page_contexts", [])
        else:
            question      = model_input["question"].iloc[0]
            page_contexts = model_input["page_contexts"].iloc[0]

        # page_contexts arrives as a JSON string from the serving endpoint
        if isinstance(page_contexts, str):
            page_contexts = _json.loads(page_contexts)

        if not page_contexts:
            return {
                "answer"    : "No page context was provided. Please navigate to a loan page and try again.",
                "pages_used": _json.dumps([]),
                "question"  : question
            }

        # Validate each page context matches the expected schema.
        # Returns a clear error instead of a silent wrong answer if
        # the Blazor side renames or drops a required field.
        try:
            from pydantic import BaseModel as _BM, ValidationError as _VE
            from typing import Any as _Any, Optional as _Opt

            class _PageCtx(_BM):
                label: str
                route: _Opt[str] = None
                structured_bucket: dict[str, _Any]
                semantic_chunks: list[str]

            for _page in page_contexts:
                _PageCtx(**_page)
        except _VE as _e:
            return {
                "answer"    : f"Invalid page context format: {_e}",
                "pages_used": _json.dumps([]),
                "question"  : question
            }

        contexts      = page_contexts[:self.MAX_PAGE_CONTEXTS]
        pages_used    = [p.get("label", "Unknown Page") for p in contexts]
        context_block = "\n\n---\n\n".join(self._format_page(p) for p in contexts)

        user_message = (
            "The lending officer is viewing the following page(s) in the loan application.\n"
            "Use only the data shown below to answer the question.\n\n"
            f"{context_block}\n\n"
            "---\n\n"
            f"Question: {question}\n\n"
            "Respond using the required format. "
            "Attribute every value to its source tab. "
            "If a value is missing from the data, say so explicitly."
        )

        response = self.client.predict(
            endpoint = self.LLM_ENDPOINT,
            inputs   = {
                "messages": [
                    {"role": "system", "content": self.SYSTEM_PROMPT},
                    {"role": "user",   "content": user_message}
                ],
                "max_tokens" : self.MAX_TOKENS,
                "temperature": 0.0
            }
        )
        answer = response["choices"][0]["message"]["content"]

        # pages_used serialised as JSON string -- MLflow schema safe
        return {
            "answer"    : answer,
            "pages_used": _json.dumps(pages_used),
            "question"  : question
        }


# -- Signature via infer_signature on typed sample dicts ----------
# page_contexts and pages_used are JSON strings so MLflow infers
# string for both -- no list-col schema issues on any MLflow version.
sample_input_sig = {
    "question"     : "What is the interest rate on this loan?",
    "page_contexts": _json.dumps([{
        "label"            : "Loan Summary",
        "structured_bucket": {"loan_number": "LN-2024-00891", "interest_rate": "6.75%"},
        "semantic_chunks"  : ["Loan status: Active"]
    }])
}
# Hardcoded sample output -- pages_used is a JSON string so MLflow
# infers string (not list), avoiding schema issues on older MLflow versions.
sample_output_sig = {
    "answer"    : "The interest rate on this loan is 6.75%.",
    "pages_used": _json.dumps(["Loan Summary"]),
    "question"  : sample_input_sig["question"]
}
signature = infer_signature(sample_input_sig, sample_output_sig)

# -- Log and register ----------------------------------------------
with mlflow.start_run(run_name="page_context_agent_registration"):

    try:
        model_info = mlflow.pyfunc.log_model(
            name                  = "page_context_agent",
            python_model          = PageContextAgentModel(),
            registered_model_name = REGISTERED_MODEL,
            signature             = signature,
            pip_requirements      = ["cloudpickle==3.0.0", "msal"],
            resources = [
                DatabricksServingEndpoint(endpoint_name="databricks-claude-haiku-4-5"),
            ],
        )
        model_uri = model_info.model_uri
    except AttributeError:
        # MLflow post-log validation bug -- model was registered successfully
        run = mlflow.active_run()
        model_uri = f"runs:/{run.info.run_id}/page_context_agent"
    print(f"Model logged: {model_uri}")

# -- Set champion alias --------------------------------------------
client_uc = mlflow.tracking.MlflowClient()
print("Waiting for model version to be ready ...")
time.sleep(10)

versions = client_uc.search_model_versions(f"name='{REGISTERED_MODEL}'")
if not versions:
    raise RuntimeError("No model versions found -- registration may have failed.")

version = versions[0].version
client_uc.set_registered_model_alias(
    name    = REGISTERED_MODEL,
    alias   = MODEL_ALIAS,
    version = version
)

print(f"Registered : {REGISTERED_MODEL}")
print(f"Version    : {version}")
print(f"Alias      : {MODEL_ALIAS}")
print("Registration complete. Proceed to Cell 8 to verify, then Cell 9 to deploy.")


In [ ]:
# ================================================================
# CELL 4 -- Deploy ace-lending-context-endpoint
# PURPOSE: Deploy ace-lending-context-endpoint as a Databricks Model Serving endpoint.
#
#
# Deploys the registered champion model as a Databricks Model
# Serving endpoint. This is the HTTP endpoint the Supervisor
# calls to answer loan-data questions.
#
# Why no mlflow.pyfunc.load_model() at serving time?
#   Calling load_model inside a container requires UC grants on
#   the registered model. Instead the model is loaded once at
#   container boot via the MLflow pyfunc runtime -- no extra
#   grants needed. Sub-agents are called as HTTP endpoints.
#
# No SP env vars injected:
#   SP credentials (client_id, secret, tenant_id) are only needed
#   for Vector Search auth (RAG agent) and SQL API inference
#   logging (Supervisor). This agent only calls the LLM, which
#   is reached via the auto-injected M2M token.
#
# Endpoint config:
#   name      : ace-lending-context-endpoint
#   model     : ace_theorem.chat.page_context_agent@champion
#   workload  : Small
#   scale_to_zero : False (keeps endpoint warm; eliminates 30-60s cold-start latency)
#
# Strategy:
#   - Endpoint missing -> POST /api/2.0/serving-endpoints (create)
#   - Endpoint exists  -> PUT  /.../config (update version)
#   - Polls every 30 s until READY, times out after 30 min.
#
# Run only after Cell 7 prints "Registration complete."
# Wait for "ace-lending-context-endpoint is READY" before
# running 05_supervisor_agent.ipynb.
# ================================================================

import requests, time, mlflow

host    = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
token   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

ENDPOINT_NAME = "ace-lending-context-endpoint"

# -- Get the latest registered version ----------------------------
mlflow.set_registry_uri("databricks-uc")
client_uc = mlflow.tracking.MlflowClient()
versions  = client_uc.search_model_versions(f"name='{REGISTERED_MODEL}'")
if not versions:
    raise RuntimeError("No model versions found. Run Cell 7 first.")
latest_version = versions[0].version
print(f"Deploying {REGISTERED_MODEL} v{latest_version} -> {ENDPOINT_NAME}")

# -- Endpoint payload ----------------------------------------------
# No SP env vars required -- only LLM endpoint access needed,
# which is granted automatically via the M2M token.
endpoint_config = {
    "served_models": [{
        "name"                 : "page-context-agent",
        "model_name"           : REGISTERED_MODEL,
        "model_version"        : latest_version,
        "workload_size"        : "Small",
        "scale_to_zero_enabled": False,
    }]
}

# -- Create or update ----------------------------------------------
check = requests.get(
    f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
    headers=headers
)

if check.status_code == 404:
    print("Endpoint does not exist -- creating ...")
    r = requests.post(
        f"{host}/api/2.0/serving-endpoints",
        headers=headers,
        json={"name": ENDPOINT_NAME, "config": endpoint_config}
    )
    r.raise_for_status()
    print("Create request accepted.")
else:
    print("Endpoint exists -- updating config ...")
    r = requests.put(
        f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
        headers=headers,
        json=endpoint_config
    )
    r.raise_for_status()
    print("Update request accepted.")

# -- Poll until READY ----------------------------------------------
print("\nPolling for READY state (max 30 min) ...")
for attempt in range(60):
    state_resp = requests.get(
        f"{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
        headers=headers
    )
    state_resp.raise_for_status()
    state         = state_resp.json().get("state", {})
    ready_state   = state.get("ready", "")
    config_update = state.get("config_update", "NOT_UPDATING")
    print(f"  [{attempt+1:02d}] ready={ready_state}  config_update={config_update}")
    if ready_state == "READY" and config_update in ("NOT_UPDATING", ""):
        break
    time.sleep(30)
else:
    raise RuntimeError(f"{ENDPOINT_NAME} did not become READY within 30 minutes.")

print("=" * 55)
print(f"Endpoint   : {ENDPOINT_NAME}")
print(f"Model      : {REGISTERED_MODEL} v{latest_version}")
print(f"Status     : READY")
print("=" * 55)
print("ace-lending-context-endpoint is READY.")
print("Proceed to 05_supervisor_agent.ipynb")